# Aspect-Based Sentiment Analysis of Hinglish Quick-Commerce Reviews with Sarcasm Detection

**Pipeline:** normalise → word-level language ID + Code-Mixing Index → aspect extraction (seed lexicon + rules) → aspect-conditioned context windows → aspect sentiment (NB / LR / Linear-SVM on char n-gram TF-IDF) → sarcasm detection (sentiment-incongruity features) → polarity correction → explanation (feature weights + LIME).

Runs on CPU (laptop or free Colab). No GPU, no paid service.

### Which data does this notebook use?
* **REAL mode** – if `data/labelled/reviews.csv` and `data/labelled/aspect_labels.csv` exist (produced from your own scraped + hand-labelled Blinkit / Zepto / Instamart reviews via `scripts/prepare_labelled_data.py`), everything below is computed on them and the numbers are reportable.
* **DEMO mode** – otherwise it falls back to `data/demo/` (149 **hand-written seed reviews** that exist to prove the code runs). **Numbers from DEMO mode must NOT be put in the report / PPT.** Every output in this mode is prefixed with `[DEMO-SYNTHETIC]`.

In REAL mode the seed reviews are used for exactly one thing: 20 of them are sarcastic, and sarcasm is so rare in real Play Store reviews (<1%) that the *deployed* sarcasm detector (section 11) is trained on real reviews + the public Swami corpus + these seed examples. They are **never** used for any evaluation number.

Two experiments always use **real public data** regardless of mode: (1) the language-ID model is trained/evaluated on L3Cube-HingLID and cross-checked on Swami et al. (2018); (2) the sarcasm detector is benchmarked on the Swami et al. (2018) sarcasm corpus (5,250 real Hinglish tweets). Run `python scripts/download_external_data.py --core-only` first.

In [1]:
# ---- 0. setup -------------------------------------------------------------
import sys, os, json, time, warnings, pathlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():          # notebook started from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90)

from absa.normalize import normalize
from absa.lid import LanguageIdentifier, read_l3cube_lid, read_swami_lid
from absa.aspects import ASPECTS, aspect_windows, detect_aspects, extract_mentions
from absa.pipeline import HinglishABSA, build_sentiment_model, build_sarcasm_model, fit_with_balance
from absa import evaluation as ev, explain as ex
from absa.features import incongruity_report

RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
MODELS = ROOT / "models"; MODELS.mkdir(exist_ok=True)
EXT = ROOT / "data" / "external"
SEED = 42; np.random.seed(SEED)
print("project root:", ROOT)

project root: /home/claude/hinglish_absa


## 1. Load the review dataset (REAL or DEMO)

In [2]:
lab_dir = ROOT / "data" / "labelled"
if (lab_dir / "reviews.csv").exists() and (lab_dir / "aspect_labels.csv").exists():
    DATA_MODE = "REAL"; data_dir = lab_dir
else:
    DATA_MODE = "DEMO-SYNTHETIC"; data_dir = ROOT / "data" / "demo"
TAG = f"[{DATA_MODE}]"
reviews = pd.read_csv(data_dir / "reviews.csv")
labels  = pd.read_csv(data_dir / "aspect_labels.csv")
labels = labels[labels["aspect"].isin(ASPECTS)].reset_index(drop=True)
reviews["rating"] = pd.to_numeric(reviews.get("rating", np.nan), errors="coerce")
reviews["sarcastic"] = reviews.get("sarcastic", 0).fillna(0).astype(int)
print(TAG, "reviews:", len(reviews), "| aspect instances:", len(labels),
      "| sarcastic reviews:", int(reviews.sarcastic.sum()), f"({reviews.sarcastic.mean():.1%})")
if DATA_MODE != "REAL":
    print("\n*** DEMO-SYNTHETIC MODE: the reviews below were written by hand to test the code. "
          "Do NOT report these metrics. Add data/labelled/ to switch to REAL mode. ***\n")
display(reviews.head())
stats = {"mode": DATA_MODE, "n_reviews": int(len(reviews)), "n_instances": int(len(labels)),
         "sarcastic_rate": float(reviews.sarcastic.mean()),
         "sentiment_dist": labels.sentiment.value_counts().to_dict(),
         "aspect_dist": labels.aspect.value_counts().to_dict()}
display(pd.crosstab(labels.aspect, labels.sentiment, margins=True))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
labels.aspect.value_counts().plot.barh(ax=ax[0], title=f"{TAG} aspect instances")
labels.sentiment.value_counts().plot.bar(ax=ax[1], title=f"{TAG} sentiment labels", rot=0)
plt.tight_layout(); plt.savefig(RESULTS / "fig_dataset.png", dpi=130); plt.show()

[REAL] reviews: 2500 | aspect instances: 3590 | sarcastic reviews: 17 (0.7%)


,review_id,text,rating,sarcastic,source
0,a401d3c3-1aeb-45e6-9ed9-c2e63c2f5cf3,The product price is less compare to other app and store. it's useful for us,5,0,zepto
1,5b6e5ad2-fd69-4d58-bf39-3a97b7297e8c,I order four item from zepto but one item was defective. defective item name is Desi F...,4,0,zepto
2,88839bd4-4dbc-4d2e-9ee4-e102bd32a96e,The app is good It's very easy for old people to use. But there's good problem that th...,4,0,blinkit
3,4ab69c17-10e6-4541-b592-4c841c057bff,"fraud company , fraud delivery partner maine ek order Kiya rs 1011 ka mujhe saman kuch...",1,0,swiggy_instamart
4,9b1396de-a896-42ab-87fb-bb940a66d65d,"Excellent app with a user-friendly interface and smooth experience. Easy to browse, or...",5,0,swiggy_instamart


sentiment,negative,neutral,positive,All
aspect,,,,
app_support,758,47,527,1332
delivery_partner,160,6,63,229
delivery_speed,231,6,344,581
general,67,54,245,366
packaging,76,8,22,106
price,268,15,183,466
product_quality,400,9,101,510
All,1960,145,1485,3590


## 2. Stage 1 — Text normalisation
Hinglish spelling is free-form. We lowercase, demojise, collapse elongations, map known spelling variants and keep `!`/`?` as tokens. The cell also lists the most frequent tokens that are **not** in our spelling map and not in any lexicon — look at that list to extend `normalize.SPELLING_MAP` for your own data.

In [3]:
examples = ["Delivery sooooo late thi, 10 min bola 1 ghanta laga 😡 bkwas service!!",
            "Wah bhai kya baat hai, doodh expired bhej diya 👏 great job",
            "Delivry boy bhut polite tha, pkging bhi acchi thi 👍"]
for s in examples:
    n = normalize(s)
    print(f"RAW : {s}\nNORM: {n.text}\n      elongations={n.n_elongations} exclaim={n.n_exclaim} emoji={n.emojis}\n")

from collections import Counter
from absa import lexicon as LX
known = set(LX.POSITIVE) | set(LX.NEGATIVE) | set(LX.NEGATORS) | set(LX.INTENSIFIERS) | \
        {t for v in LX.ASPECT_LEXICON.values() for t in v} | set(LX.SARCASM_MARKERS)
STOP = set("to the and i is it a for me in they but this my you was are with have from that on when or also your all be get more if of as at by so not no we he she his her them their there here what which who how do does did can could would should will just very only even than then too one any some about out up now its im am been being has had our us ko ka ki ke se ne mein par pe bhi hi nahi na ho hain hoon tha thi ye yeh wo woh koi kuch kar karo karte karta karti kiya kya kyu aur ya ab abhi phir fir jab tab jo jis hota hoti hote raha rahi rahe diya di de do le lo liya li mil mila milta aa aaya aati aata gaya gayi gaye bahut sab app aap apne apna apni mere mera meri hum mujhe main mai tum ek use".split())
cnt = Counter(t for txt in reviews.text for t in normalize(txt).tokens
              if t not in known and t not in STOP and t.isalpha() and len(t) > 2)
print(TAG, "most frequent tokens outside all lexicons (candidates for the spelling map / lexicon):")
print(cnt.most_common(40))

RAW : Delivery sooooo late thi, 10 min bola 1 ghanta laga 😡 bkwas service!!
NORM: delivery soo late thi ten_minute bola <num> ghanta laga emoji_enraged_face bakwas service !
      elongations=1 exclaim=2 emoji=['enraged_face']

RAW : Wah bhai kya baat hai, doodh expired bhej diya 👏 great job
NORM: wah_bhai kya_baat doodh expired bhej diya emoji_clapping_hands great job
      elongations=0 exclaim=0 emoji=['clapping_hands']

RAW : Delivry boy bhut polite tha, pkging bhi acchi thi 👍
NORM: delivery_partner bahut polite tha packaging bhi accha thi emoji_thumbs_up
      elongations=0 exclaim=0 emoji=['thumbs_up']



[REAL] most frequent tokens outside all lexicons (candidates for the spelling map / lexicon):
[('order', 1024), ('hai', 679), ('delivery', 660), ('zepto', 310), ('swiggy', 298), ('service', 298), ('time', 281), ('food', 253), ('blinkit', 227), ('product', 201), ('experience', 201), ('items', 179), ('after', 149), ('products', 145), ('like', 144), ('please', 121), ('other', 116), ('need', 106), ('item', 105), ('want', 105), ('give', 104), ('customer', 101), ('always', 95), ('money', 88), ('times', 88), ('location', 87), ('able', 81), ('help', 79), ('better', 79), ('buy', 79), ('return', 75), ('because', 73), ('received', 73), ('much', 72), ('add', 72), ('team', 68), ('call', 67), ('purchase', 67), ('make', 65), ('see', 64)]


## 3. Stage 2 — Word-level language identification + Code-Mixing Index
A char n-gram (1–4) TF-IDF + logistic regression classifier trained on **L3Cube-HingLID** (real Hinglish tweets, CC BY-NC-SA 4.0). We evaluate on its held-out test split and *cross-dataset* on the word-level tags of **Swami et al. (2018)**. Then we compute the CMI of every review in our dataset.

In [4]:
t0 = time.time()
w_tr, l_tr = read_l3cube_lid(str(EXT / "l3cube_hinglid_train.txt"), max_tokens=200_000)
lid = LanguageIdentifier().fit(w_tr, l_tr)
w_te, l_te = read_l3cube_lid(str(EXT / "l3cube_hinglid_test.txt"))
acc_l3 = float(np.mean(lid.model.predict(w_te) == np.array(l_te)))
w_sw, l_sw = read_swami_lid(str(EXT / "swami_lid_tags.txt"))
acc_sw = float(np.mean(lid.model.predict(w_sw) == np.array(l_sw)))
print(f"[REAL PUBLIC DATA] LID trained on {len(w_tr):,} tokens in {time.time()-t0:.1f}s")
print(f"  L3Cube-HingLID test accuracy  : {acc_l3:.4f}  (n={len(w_te):,} tokens)")
print(f"  Swami et al. cross-dataset acc: {acc_sw:.4f}  (n={len(w_sw):,} tokens)")
lid.save(MODELS / "lid.joblib")
lid_metrics = {"l3cube_test_acc": acc_l3, "swami_cross_acc": acc_sw, "train_tokens": len(w_tr)}

# CMI of our reviews
cmis = [lid.analyse(normalize(t).tokens).cmi for t in reviews.text]
reviews["cmi"] = cmis
print(f"\n{TAG} CMI: mean={np.mean(cmis):.1f}, median={np.median(cmis):.1f}, "
      f"monolingual (CMI=0): {np.mean(np.array(cmis)==0):.1%}")
plt.figure(figsize=(6,3)); plt.hist(cmis, bins=20); plt.title(f"{TAG} Code-Mixing Index per review")
plt.xlabel("CMI (0 = monolingual, 50 = perfectly mixed)"); plt.tight_layout()
plt.savefig(RESULTS / "fig_cmi.png", dpi=130); plt.show()
n = normalize(examples[0]); r = lid.analyse(n.tokens)
print(list(zip(r.tokens, r.langs)), "CMI =", r.cmi)

[REAL PUBLIC DATA] LID trained on 200,000 tokens in 3.8s
  L3Cube-HingLID test accuracy  : 0.9625  (n=190,518 tokens)
  Swami et al. cross-dataset acc: 0.9160  (n=95,322 tokens)



[REAL] CMI: mean=17.1, median=13.6, monolingual (CMI=0): 1.2%
[('delivery', 'en'), ('soo', np.str_('en')), ('late', np.str_('en')), ('thi', 'hi'), ('ten_minute', 'en'), ('bola', np.str_('hi')), ('<num>', 'other'), ('ghanta', np.str_('hi')), ('laga', np.str_('hi')), ('emoji_enraged_face', 'other'), ('bakwas', np.str_('hi')), ('service', np.str_('en')), ('!', 'other')] CMI = 50.0


## 4. Stage 3 — Aspect extraction (seed lexicon + rules) and context windows
We check the rule-based extractor against the gold aspects (multi-label precision / recall / F1 per aspect) and show the aspect-conditioned windows.

In [5]:
asp_rep = ev.aspect_extraction_report(reviews, labels)
print(TAG, "aspect extraction vs gold labels"); display(asp_rep)
asp_rep.to_csv(RESULTS / "aspect_extraction.csv", index=False)
for s in ["delivery fast thi but doodh expired tha, packing bhi thik thi",
          "Price thoda mehnga hai lekin delivery boy bahut polite tha aur app smooth chalta hai"]:
    toks = normalize(s).tokens
    print("\n", s); print("  mentions:", [(m.aspect, m.trigger) for m in extract_mentions(toks)])
    for k in (3, 5):
        print(f"  windows k={k}:", aspect_windows(toks, k=k))
    print("  no-window :", aspect_windows(toks, use_window=False))

[REAL] aspect extraction vs gold labels


,aspect,support,tp,fp,fn,precision,recall,f1
0,delivery_speed,581,477,140,104,0.773,0.821,0.796
1,product_quality,510,360,103,150,0.778,0.706,0.740
2,packaging,106,96,43,10,0.691,0.906,0.784
3,price,466,425,125,41,0.773,0.912,0.837
4,delivery_partner,229,194,81,35,0.705,0.847,0.770
5,app_support,1332,1081,324,251,0.769,0.812,0.790
6,general,366,204,242,162,0.457,0.557,0.502
7,MICRO-AVG,3590,2837,1058,753,0.728,0.790,0.758



 delivery fast thi but doodh expired tha, packing bhi thik thi
  mentions: [('delivery_speed', 'fast_delivery'), ('product_quality', 'doodh'), ('product_quality', 'expired'), ('packaging', 'packaging')]
  windows k=3: {'delivery_speed': 'fast_delivery thi', 'product_quality': 'doodh expired tha', 'packaging': 'tha packaging bhi theek thi'}
  windows k=5: {'delivery_speed': 'fast_delivery thi', 'product_quality': 'doodh expired tha', 'packaging': 'tha packaging bhi theek thi'}
  no-window : {'delivery_speed': 'fast_delivery thi but doodh expired tha packaging bhi theek thi', 'product_quality': 'fast_delivery thi but doodh expired tha packaging bhi theek thi', 'packaging': 'fast_delivery thi but doodh expired tha packaging bhi theek thi'}

 Price thoda mehnga hai lekin delivery boy bahut polite tha aur app smooth chalta hai
  mentions: [('price', 'price'), ('price', 'mehenga'), ('delivery_partner', 'delivery_partner'), ('delivery_partner', 'polite'), ('app_support', 'app')]
  windows k=

## 5. Stage 4 — Aspect-conditioned sentiment classification: three baselines
For every gold (review, aspect) pair we build the window text and train three classifiers on char(2–5)+word(1–2) TF-IDF + 7 window-lexicon counts. Class imbalance → `class_weight='balanced'` (LR, SVM) or balanced `sample_weight` (NB). Evaluation → **stratified group 5-fold CV** (all windows of one review stay in one fold). Macro-F1 is the headline metric because the classes are imbalanced.

In [6]:
system = HinglishABSA(lid=lid, window_k=5)
inst = system.build_instances(reviews, labels)
print(TAG, f"{len(inst)} instances; aspect found by lexicon for {inst.found.mean():.1%} (others use full text)")
sent_results, oof = {}, {}
for kind in ["nb", "lr", "svm"]:
    t0 = time.time()
    res = ev.cv_sentiment(inst, lambda k=kind: build_sentiment_model(k), kind, seed=SEED)
    oof[kind] = res.pop("oof_pred"); res["time_s"] = round(time.time() - t0, 1)
    sent_results[kind] = res
tbl = pd.DataFrame(sent_results).T.rename(index={"nb": "ComplementNB", "lr": "LogisticRegression", "svm": "LinearSVM(calibrated)"})
print(TAG, "aspect sentiment — 5-fold stratified group CV"); display(tbl)
tbl.to_csv(RESULTS / "sentiment_baselines.csv")
BEST = max(sent_results, key=lambda k: sent_results[k]["f1_macro"])
print("best by macro-F1:", BEST)

[REAL] 3590 instances; aspect found by lexicon for 79.0% (others use full text)


[REAL] aspect sentiment — 5-fold stratified group CV


,accuracy,precision_macro,recall_macro,f1_macro,f1_macro_std,n_splits,time_s
ComplementNB,0.8691,0.7126,0.7187,0.7155,0.0349,5.0,2.1
LogisticRegression,0.8799,0.7348,0.7047,0.7171,0.0358,5.0,6.2
LinearSVM(calibrated),0.8813,0.7743,0.6314,0.6394,0.0124,5.0,7.0


best by macro-F1: lr


In [7]:
print(TAG, f"per-aspect results ({BEST})"); per_aspect = ev.per_aspect_table(inst, oof[BEST]); display(per_aspect)
per_aspect.to_csv(RESULTS / "per_aspect.csv", index=False)
cm = ev.confusion(inst.sentiment, oof[BEST]); display(cm)
print(ev.report_text(inst.sentiment, oof[BEST]))
fig, ax = plt.subplots(figsize=(4.5, 4)); im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels(ev.SENTIMENT_ORDER); ax.set_yticklabels(ev.SENTIMENT_ORDER)
for i in range(3):
    for j in range(3): ax.text(j, i, cm.values[i, j], ha="center", va="center", color="k")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{TAG} confusion ({BEST})")
plt.tight_layout(); plt.savefig(RESULTS / "fig_confusion.png", dpi=130); plt.show()

[REAL] per-aspect results (lr)


,aspect,n,accuracy,precision_macro,recall_macro,f1_macro
0,app_support,1332,0.9039,0.7409,0.7216,0.7302
2,delivery_speed,581,0.9191,0.7257,0.7257,0.7257
6,product_quality,510,0.8941,0.6094,0.6234,0.6160
5,price,466,0.8004,0.6033,0.6101,0.6064
3,general,366,0.8279,0.7688,0.7338,0.7184
1,delivery_partner,229,0.8690,0.5672,0.5718,0.5693
4,packaging,106,0.8491,0.6975,0.6896,0.6925


,pred_negative,pred_neutral,pred_positive
true_negative,1797,41,122
true_neutral,60,45,40
true_positive,146,22,1317


              precision    recall  f1-score   support

    negative      0.897     0.917     0.907      1960
     neutral      0.417     0.310     0.356       145
    positive      0.890     0.887     0.889      1485

    accuracy                          0.880      3590
   macro avg      0.735     0.705     0.717      3590
weighted avg      0.875     0.880     0.877      3590



## 6. Ablation A — do aspect windows help?
Same classifier, different inputs: full review text (no window) vs windows of size 3 / 5 / 8, with and without stopping at clause breakers, and with/without the 7 lexicon features.

In [8]:
abl_rows = []
configs = [("no window (full text)", dict(use_window=False)),
           ("window k=3", dict(window_k=3)), ("window k=5", dict(window_k=5)), ("window k=8", dict(window_k=8)),
           ("window k=5, no clause stop", dict(window_k=5, stop_at_clause=False)),
           ("window k=5, no lexicon feats", dict(window_k=5, use_lexicon_feats=False))]
for name, kw in configs:
    sysm = HinglishABSA(lid=lid, **{k: v for k, v in kw.items() if k != "use_lexicon_feats"})
    ins = sysm.build_instances(reviews, labels)
    ulf = kw.get("use_lexicon_feats", True)
    res = ev.cv_sentiment(ins, lambda k=BEST, u=ulf: build_sentiment_model(k, use_lexicon_feats=u), BEST, seed=SEED)
    abl_rows.append({"setting": name, "f1_macro": res["f1_macro"], "±std": res["f1_macro_std"],
                     "accuracy": res["accuracy"], "precision_macro": res["precision_macro"], "recall_macro": res["recall_macro"]})
ablation_windows = pd.DataFrame(abl_rows); print(TAG, f"window ablation ({BEST})"); display(ablation_windows)
ablation_windows.to_csv(RESULTS / "ablation_windows.csv", index=False)

# Windows can only matter where a review carries MORE THAN ONE aspect (otherwise window ~= full text).
multi = labels.groupby("review_id").aspect.nunique(); multi_ids = set(multi[multi >= 2].index)
rows = []
for name, kw in [("no window (full text)", dict(use_window=False)), ("window k=5", dict(window_k=5))]:
    sysm = HinglishABSA(lid=lid, **kw); ins = sysm.build_instances(reviews, labels)
    res = ev.cv_sentiment(ins, lambda k=BEST: build_sentiment_model(k), BEST, seed=SEED)
    m = ins.review_id.isin(multi_ids).to_numpy()
    sub = ev.summarise(ins.sentiment[m], res["oof_pred"][m], labels=ev.SENTIMENT_ORDER) if m.any() else {}
    rows.append({"setting": name, "multi_aspect_instances": int(m.sum()), **{f"multi_{k}": v for k, v in sub.items()}})
ablation_multi = pd.DataFrame(rows); print(TAG, "same ablation restricted to multi-aspect reviews"); display(ablation_multi)
ablation_multi.to_csv(RESULTS / "ablation_windows_multiaspect.csv", index=False)

[REAL] window ablation (lr)


,setting,f1_macro,±std,accuracy,precision_macro,recall_macro
0,no window (full text),0.7348,0.0352,0.8877,0.7374,0.7324
1,window k=3,0.6943,0.0359,0.8694,0.7177,0.6804
2,window k=5,0.7171,0.0358,0.8799,0.7348,0.7047
3,window k=8,0.7197,0.0434,0.8827,0.7344,0.7087
4,"window k=5, no clause stop",0.7098,0.0296,0.8758,0.7320,0.6956
5,"window k=5, no lexicon feats",0.6972,0.0143,0.8786,0.7320,0.6795


[REAL] same ablation restricted to multi-aspect reviews


,setting,multi_aspect_instances,multi_accuracy,multi_precision_macro,multi_recall_macro,multi_f1_macro
0,no window (full text),2022,0.9130,0.6108,0.6122,0.6115
1,window k=5,2022,0.9001,0.6458,0.6679,0.6528


## 7. Stage 5 — Sarcasm detection with sentiment-incongruity features
**7a. Public benchmark (real data).** Swami et al. (2018): 5,250 Hinglish tweets, 504 sarcastic (~10%). We compare text-only TF-IDF, incongruity-only, and both, for NB / LR / SVM with 5-fold stratified CV. The original paper reports 78.4% F1 with hand-crafted features + random forest.

**7b. Our review dataset.** Same models on the reviews (here the `rating_text_clash` feature is available, which tweets do not have).

In [9]:
swami = pd.read_csv(EXT / "swami_sarcasm.csv")
print(f"[REAL PUBLIC DATA] Swami et al. 2018: {len(swami)} tweets, {swami.sarcastic.sum()} sarcastic")
rows = []
for kind in ["nb", "lr", "svm"]:
    for fname, kw in [("text only", dict(use_incongruity=False)), ("incongruity only", dict(use_text=False)), ("text + incongruity", {})]:
        t0 = time.time()
        res = ev.cv_sarcasm(swami, lambda k=kind, kw=kw: build_sarcasm_model(k, lid=lid, **kw), kind, seed=SEED)
        rows.append({"model": kind, "features": fname, "f1_sarcastic": res["f1_sarcastic"], "precision_sarcastic": res["precision_sarcastic"],
                     "recall_sarcastic": res["recall_sarcastic"], "f1_macro": res["f1_macro"], "accuracy": res["accuracy"], "time_s": round(time.time()-t0, 1)})
swami_tbl = pd.DataFrame(rows); print("[REAL PUBLIC DATA] sarcasm on Swami et al. (5-fold CV)"); display(swami_tbl)
swami_tbl.to_csv(RESULTS / "sarcasm_swami_benchmark.csv", index=False)

[REAL PUBLIC DATA] Swami et al. 2018: 5250 tweets, 504 sarcastic


[REAL PUBLIC DATA] sarcasm on Swami et al. (5-fold CV)


,model,features,f1_sarcastic,precision_sarcastic,recall_sarcastic,f1_macro,accuracy,time_s
0,nb,text only,0.7471,0.6433,0.8909,0.8572,0.9421,22.1
1,nb,incongruity only,0.1958,0.1141,0.6885,0.3931,0.4571,38.0
2,nb,text + incongruity,0.7446,0.6427,0.8849,0.8558,0.9417,61.0
3,lr,text only,0.7831,0.6719,0.9385,0.8775,0.9501,22.7
4,lr,incongruity only,0.2397,0.1521,0.5655,0.5085,0.6556,38.2
5,lr,text + incongruity,0.7901,0.6751,0.9524,0.8813,0.9514,61.3
6,svm,text only,0.7855,0.6818,0.9266,0.8791,0.9514,23.9
7,svm,incongruity only,0.0039,0.2500,0.0020,0.4766,0.9036,38.0
8,svm,text + incongruity,0.7848,0.6828,0.9226,0.8787,0.9514,63.6


In [10]:
rows, sarc_oof = [], {}
n_sarc = int(reviews.sarcastic.sum())
print(TAG, f"{n_sarc} sarcastic reviews out of {len(reviews)} ({n_sarc/len(reviews):.1%})")
if n_sarc >= 5:
    feats = [("text only", dict(use_incongruity=False)), ("incongruity only", dict(use_text=False)), ("text + incongruity", {})]
    # (a) reviews only;  (b) TRANSFER: Swami tweets added to every training fold (tested on reviews only)
    for train_set, extra in [("reviews only", None), ("reviews + Swami (transfer)", swami)]:
        for kind in (["nb", "lr", "svm"] if extra is None else ["nb", "lr"]):
            for fname, kw in feats:
                res = ev.cv_sarcasm(reviews, lambda k=kind, kw=kw: build_sarcasm_model(k, lid=lid, **kw), kind, seed=SEED, extra_train=extra)
                sarc_oof[(train_set, kind, fname)] = res["oof_pred"]
                rows.append({"train_set": train_set, "model": kind, "features": fname, "f1_sarcastic": res["f1_sarcastic"],
                             "precision_sarcastic": res["precision_sarcastic"], "recall_sarcastic": res["recall_sarcastic"],
                             "f1_macro": res["f1_macro"], "accuracy": res["accuracy"]})
    sarc_tbl = pd.DataFrame(rows); print(TAG, "sarcasm on the review dataset (5-fold CV, tested on reviews only)"); display(sarc_tbl)
    sarc_tbl.to_csv(RESULTS / "sarcasm_reviews.csv", index=False)
    SARC_BEST = sarc_tbl.sort_values(["f1_sarcastic", "f1_macro"], ascending=False).iloc[0]
    print("best:", SARC_BEST.train_set, "/", SARC_BEST.model, "/", SARC_BEST.features)
    if n_sarc < 50:
        print(f"\nNOTE: only {n_sarc} positive examples -> these review-level sarcasm numbers are unstable; "
              "the public Swami benchmark above is the reliable measure of the detector. See the limitations section.")
else:
    print(TAG, "too few sarcastic reviews to cross-validate a sarcasm model"); SARC_BEST = None

[REAL] 17 sarcastic reviews out of 2500 (0.7%)


[REAL] sarcasm on the review dataset (5-fold CV, tested on reviews only)


,train_set,model,features,f1_sarcastic,precision_sarcastic,recall_sarcastic,f1_macro,accuracy
0,reviews only,nb,text only,0.0656,0.0455,0.1176,0.5270,0.9772
1,reviews only,nb,incongruity only,0.0352,0.0181,0.5882,0.4558,0.7808
2,reviews only,nb,text + incongruity,0.0714,0.0513,0.1176,0.5305,0.9792
3,reviews only,lr,text only,0.0000,0.0000,0.0000,0.4979,0.9916
4,reviews only,lr,incongruity only,0.0403,0.0209,0.5882,0.4673,0.8096
5,reviews only,lr,text + incongruity,0.1667,0.2857,0.1176,0.5813,0.9920
6,reviews only,svm,text only,0.0000,0.0000,0.0000,0.4983,0.9932
7,reviews only,svm,incongruity only,0.0000,0.0000,0.0000,0.4983,0.9932
8,reviews only,svm,text + incongruity,0.0000,0.0000,0.0000,0.4982,0.9928
9,reviews + Swami (transfer),nb,text only,0.0435,0.0345,0.0588,0.5173,0.9824


best: reviews only / lr / text + incongruity

NOTE: only 17 positive examples -> these review-level sarcasm numbers are unstable; the public Swami benchmark above is the reliable measure of the detector. See the limitations section.


## 8. Ablation B — polarity correction: with vs without sarcasm handling
End-to-end: the out-of-fold aspect-sentiment predictions are corrected with the out-of-fold sarcasm predictions (positive → negative when the review is predicted sarcastic). We measure on all instances and on the sarcastic subset.

In [11]:
if SARC_BEST is not None:
    sarc_pred = dict(zip(reviews.review_id, sarc_oof[(SARC_BEST.train_set, SARC_BEST.model, SARC_BEST.features)]))
    base = list(oof[BEST])
    corrected = ev.apply_correction(base, sarc_pred, inst.review_id.tolist())
    gold_sarc = dict(zip(reviews.review_id, reviews.sarcastic))
    mask = inst.review_id.map(gold_sarc).astype(bool).to_numpy()
    rows = []
    for name, pred in [("without sarcasm handling", base), ("with sarcasm handling (predicted)", corrected),
                       ("with sarcasm handling (oracle gold flags)", ev.apply_correction(base, gold_sarc, inst.review_id.tolist()))]:
        all_m = ev.summarise(inst.sentiment, pred, labels=ev.SENTIMENT_ORDER)
        sub_m = ev.summarise(inst.sentiment[mask], np.array(pred)[mask], labels=ev.SENTIMENT_ORDER) if mask.any() else {}
        rows.append({"setting": name, "all_f1_macro": all_m["f1_macro"], "all_accuracy": all_m["accuracy"],
                     "sarcastic_subset_accuracy": sub_m.get("accuracy"), "sarcastic_subset_n": int(mask.sum())})
    ablation_sarcasm = pd.DataFrame(rows); print(TAG, "polarity-correction ablation"); display(ablation_sarcasm)
    ablation_sarcasm.to_csv(RESULTS / "ablation_sarcasm.csv", index=False)

[REAL] polarity-correction ablation


,setting,all_f1_macro,all_accuracy,sarcastic_subset_accuracy,sarcastic_subset_n
0,without sarcasm handling,0.7171,0.8799,0.6842,19
1,with sarcasm handling (predicted),0.7169,0.8797,0.6842,19
2,with sarcasm handling (oracle gold flags),0.7179,0.8811,0.8947,19


## 9. Explainability — feature-weight inspection and LIME
* **Global:** the strongest n-grams / lexicon features per class (what did the model learn?).
* **Local, exact:** per-feature contribution `weight × value` for one input (linear model).
* **Local, model-agnostic:** LIME word-level explanation.

In [12]:
# Deployed model. Aspect sentiment: trained on the review data only.
# Sarcasm: real reviews + public Swami corpus (5,250 tweets) + 149 hand-written seed examples
# (data/demo, 20 sarcastic) -- the seed examples are TRAINING-ONLY augmentation for a <1% class
# and never enter any evaluation number above.
seed_df = pd.read_csv(ROOT / "data" / "demo" / "reviews.csv")[["text", "rating", "sarcastic"]]
extra_sarc = pd.concat([swami[["text", "sarcastic"]], seed_df], ignore_index=True) if DATA_MODE == "REAL" else None
final = HinglishABSA(lid=lid, window_k=5, sentiment_kind=BEST, sarcasm_kind="lr").fit(reviews, labels, extra_sarcasm=extra_sarc)
print("sarcasm model trained on", final.meta.get("sarcasm_extra_train", 0), "extra examples +", len(reviews), "reviews")
print(TAG, "top features per class (global):"); display(ex.top_features(final.sentiment_model, n=8))
ex.top_features(final.sentiment_model, n=15).to_csv(RESULTS / "top_features_sentiment.csv", index=False)
demo_text = "Wah bhai kya baat hai, doodh expired bhej diya 👏 great job"
r = final.predict_review(demo_text, rating=1)
print("\nINPUT:", demo_text); print("sarcastic:", r.sarcastic, f"(p={r.sarcasm_proba:.2f})  CMI={r.cmi}")
for a in r.aspects:
    print(f"  {a.aspect:16s} window='{a.window}'  raw={a.sentiment_raw}  final={a.sentiment_final}  corrected={a.corrected}")
print("\nexact contributions for the window (towards predicted class):")
display(ex.contributions(final.sentiment_model, r.aspects[0].window, n=8))
if final.sarcasm_model is not None:
    print("\nsarcasm model contributions (towards 'sarcastic'):")
    display(ex.contributions(final.sarcasm_model, pd.DataFrame([{"text": demo_text, "rating": 1}]), target_class=1, n=8))
pred, lime_list = ex.lime_explain(final.sentiment_model, r.aspects[0].window, num_samples=300)
print("\nLIME (word level) for predicted class", pred, ":", lime_list)

sarcasm model trained on 5399 extra examples + 2500 reviews
[REAL] top features per class (global):


,class,rank,feature,weight
0,negative,1,lex__win_neg_max,2.314988
1,negative,2,text__word__not,1.280092
2,negative,3,text__word__not good,1.091075
3,negative,4,text__word__high,1.017195
4,negative,5,text__char__ed,0.882344
5,negative,6,text__char__not,0.767192
6,negative,7,text__char__not,0.742253
7,negative,8,text__word__to,0.733043
8,neutral,1,text__word__to you,1.506871
9,neutral,2,text__word__please,1.479351



INPUT: Wah bhai kya baat hai, doodh expired bhej diya 👏 great job
sarcastic: True (p=0.99)  CMI=37.5
  product_quality  window='wah_bhai kya_baat doodh expired bhej diya emoji_clapping_hands great job'  raw=positive  final=negative  corrected=True

exact contributions for the window (towards predicted class):


,feature,value,weight,contribution,towards
0,lex__win_pos_max,0.900000,1.928248,1.735423,positive
1,lex__win_neg_max,1.000000,-1.432650,-1.432650,positive
2,lex__win_pos_count,4.000000,0.227460,0.909841,positive
3,lex__win_len,0.300000,-1.013910,-0.304173,positive
4,text__word__great,0.205847,0.493272,0.101539,positive
5,lex__win_neg_count,1.000000,0.057223,0.057223,positive
6,text__word__diya,0.256197,-0.135599,-0.034740,positive
7,text__word__job,0.325565,0.099704,0.032460,positive



sarcasm model contributions (towards 'sarcastic'):


,feature,value,weight,contribution,towards
0,incong__rating_text_clash,1.000000,2.086075,2.086075,1
1,incong__marker_count,0.625000,1.612083,1.007552,1
2,incong__log_len,0.330188,2.937332,0.969873,1
3,incong__incongruity_strength,0.900000,0.712421,0.641179,1
4,incong__marker_with_neg,1.000000,0.572051,0.572051,1
5,incong__incongruity_flag,1.000000,-0.544836,-0.544836,1
6,incong__cmi,0.750000,0.422108,0.316581,1
7,text__word__kya_baat,0.325195,0.778991,0.253324,1



LIME (word level) for predicted class positive : [(np.str_('expired'), -0.3968415017377985), (np.str_('great'), 0.27099630351089643), (np.str_('diya'), -0.09250908286781265), (np.str_('emoji_clapping_hands'), 0.07755080167806286), (np.str_('kya_baat'), 0.07733095701440652), (np.str_('wah_bhai'), 0.06278869128341921), (np.str_('bhej'), -0.04557933059662161), (np.str_('doodh'), 0.019836569906979212)]


## 10. Error analysis — real misclassified examples
Out-of-fold errors of the best sentiment model, plus sarcasm errors. The last column gives an automatic first guess at the error type; read each row and refine the category by hand for the report.

In [13]:
err = ev.error_table(inst, oof[BEST], n=40)
def guess(row):
    toks = row.window.split()
    if not row.found: return "aspect not found by lexicon → full text used"
    if any(t in LX.NEGATORS for t in toks): return "negation inside window"
    if row.sentiment == "neutral" or row.pred == "neutral": return "neutral vs polar confusion"
    if not any(t in LX.POSITIVE or t in LX.NEGATIVE for t in toks): return "no sentiment word in window (implicit sentiment)"
    return "mixed / other"
err["error_type"] = err.apply(guess, axis=1)
print(TAG, f"{len(err)} shown of {int((inst.sentiment != oof[BEST]).sum())} sentiment errors")
display(err[["aspect", "sentiment", "pred", "window", "error_type"]])
err.to_csv(RESULTS / "errors_sentiment.csv", index=False)
print(err.error_type.value_counts())
if SARC_BEST is not None:
    se = reviews.assign(pred=sarc_oof[(SARC_BEST.train_set, SARC_BEST.model, SARC_BEST.features)])
    se = se[se.sarcastic != se.pred][["review_id", "text", "rating", "sarcastic", "pred"]]
    print(f"\n{TAG} sarcasm errors ({len(se)}):"); display(se.head(20)); se.to_csv(RESULTS / "errors_sarcasm.csv", index=False)

[REAL] 40 shown of 431 sentiment errors


,aspect,sentiment,pred,window,error_type
0,price,positive,negative,the product price is less compare to other,no sentiment word in window (implicit sentiment)
2,delivery_partner,negative,positive,there's good problem that the partners have responsible for dutvy,mixed / other
18,product_quality,negative,positive,are also not of good quality,negation inside window
25,price,positive,negative,and less price compared to other online vendor,no sentiment word in window (implicit sentiment)
30,general,neutral,positive,first time hai mera is app se,aspect not found by lexicon → full text used
35,general,positive,negative,overall good experience with zepto but i don t recommend buying milkymist paneer from ...,aspect not found by lexicon → full text used
38,delivery_speed,positive,negative,mai kaha tha ki <num> minute mai ayga yeto minute mai agaya,no sentiment word in window (implicit sentiment)
46,delivery_speed,positive,negative,accha hai delivery sahi time pr krte hai payment ka b koi issue nahi hai delivery_part...,aspect not found by lexicon → full text used
47,app_support,positive,negative,sahi time pr krte hai payment ka b koi issue nahi hai,negation inside window
56,price,negative,positive,handeling charge thoda kam rakho overall best,mixed / other


error_type
aspect not found by lexicon → full text used        10
no sentiment word in window (implicit sentiment)     9
neutral vs polar confusion                           9
negation inside window                               8
mixed / other                                        4
Name: count, dtype: int64

[REAL] sarcasm errors (20):


,review_id,text,rating,sarcastic,pred
136,c1f75fd7-f33f-4f00-b528-ec4496735c33,handling fee?? abb delivery boy laa Raha hai to handle to Karega hi na😏😏,4,1,0
198,17aaeecb-19bb-40b3-83dc-853e46a7e6a3,199 ₹purchase and get complimentary 1 ₹.. fake. we can make bill 500 to 600 but 1₹ off...,2,1,0
312,30ff4f76-3fa8-438c-bc08-62ce6d46be63,2 rupees se ab 13 rupees surge charge wow bhai blinkit 50-100 rupee lelo extra charge ...,1,1,0
324,b2381a5b-3edd-4c38-9834-55537cf30401,"Giving offer and during payment price sudden change, Ek dum se Waqt badal diye, Jasbaa...",5,1,0
476,b6b55712-79e0-4305-8132-c58e289525ab,Thank you Swiggy. Aap k wajah sei aaj mei vhuka sone wala hu.,1,1,0
671,9b4cdeb9-7d35-466c-b7a9-2dbadfde85f1,i select the product to buy.. And after 10 minutes blinkit automatically increases the...,3,1,0
893,ef036d88-76b8-44c5-829d-ce0d6568d22c,had a horrible experience with Instamart and even after raising a query for like 10 ti...,2,1,0
1088,37f48d27-e8cd-41e8-a693-f2ed5b96548b,handling fees lene lag gya agya na apni aukaat pe😹😹,1,1,0
1172,edc11637-b902-4250-8f06-9942c367875e,What a revolutionary app! The creators were able to use Einstein's special relativity ...,2,1,0
1228,0cb29365-4a43-4ea1-ad23-7d0722429b98,Swiggy have ability to stop the time midway between estimated time and that's why they...,2,1,0


## 11. Save the final models + a metrics summary (used by the Streamlit app and the report)

In [14]:
final.meta.update({"data_mode": DATA_MODE, "best_sentiment_model": BEST,
                   "best_sarcasm_model": "lr/text+incongruity (reviews + Swami + seed examples)",
                   "best_sarcasm_cv": None if SARC_BEST is None else f"{SARC_BEST.train_set}/{SARC_BEST.model}/{SARC_BEST.features}",
                   "trained_on": str(data_dir.name)})
final.save(MODELS / "absa_model.joblib")
summary = {"data_mode": DATA_MODE, "dataset": stats, "lid": lid_metrics, "sentiment_cv": sent_results, "best_sentiment": BEST,
           "window_ablation": ablation_windows.to_dict("records"), "swami_sarcasm_benchmark": swami_tbl.to_dict("records"),
           "sarcasm_reviews": (sarc_tbl.to_dict("records") if SARC_BEST is not None else None),
           "polarity_correction_ablation": (ablation_sarcasm.to_dict("records") if SARC_BEST is not None else None)}
(RESULTS / "metrics.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved models/absa_model.joblib, models/lid.joblib, results/metrics.json")
if DATA_MODE != "REAL":
    print("\n*** Reminder: this run used SYNTHETIC demo data. Only the L3Cube-LID and Swami-sarcasm numbers are real. ***")

saved models/absa_model.joblib, models/lid.joblib, results/metrics.json


## 12. (Optional) Transformer comparison — why it is optional and when to run it
A pretrained multilingual transformer (e.g. `google/muril-base-cased` or `xlm-roberta-base`) can be fine-tuned on the same windows. **We keep it optional because:** (1) the project constraint is CPU-only and 1,500–3,000 labelled windows — a 110M-parameter model fine-tuned on this little data on CPU takes 30–60 min per fold and typically gains little over char n-gram linear models (SentiMix 2020 top systems used BERT variants but the margin over strong linear baselines was modest); (2) linear models give exact explanations; (3) it needs an internet download (~1 GB). Run the cell below on Colab only if you want the comparison for the report; it reuses the same CV folds so the numbers are comparable. Leave `RUN_TRANSFORMER=False` otherwise.

In [15]:
RUN_TRANSFORMER = False   # set True on Colab (CPU ok but slow; GPU faster) to add the comparison row
if RUN_TRANSFORMER:
    # pip install transformers torch sentence-transformers  (not in requirements.txt on purpose)
    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedGroupKFold
    # Cheap variant: frozen multilingual embeddings + LR (no fine-tuning, runs on CPU in a few minutes)
    st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    Xe = st.encode(inst.window.tolist(), batch_size=32, show_progress_bar=False)
    y = inst.sentiment.to_numpy(); g = inst.review_id.to_numpy()
    oof_t = np.empty(len(y), dtype=object)
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(Xe, y, g):
        oof_t[te] = LogisticRegression(class_weight="balanced", max_iter=3000).fit(Xe[tr], y[tr]).predict(Xe[te])
    print(TAG, "frozen multilingual embeddings + LR:", ev.summarise(y, oof_t, labels=ev.SENTIMENT_ORDER))
else:
    print("transformer comparison skipped (RUN_TRANSFORMER=False)")

transformer comparison skipped (RUN_TRANSFORMER=False)
